In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **UNET**

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_op = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv_op(x)


class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        down = self.conv(x)
        p = self.pool(down)
        return down, p


class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
       x1 = self.up(x1)
       x = torch.cat([x1, x2], 1)
       return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.down_convolution_1 = DownSample(in_channels, 64)
        self.down_convolution_2 = DownSample(64, 128)
        self.down_convolution_3 = DownSample(128, 256)
        self.down_convolution_4 = DownSample(256, 512)

        self.bottle_neck = DoubleConv(512, 1024)

        self.up_convolution_1 = UpSample(1024, 512)
        self.up_convolution_2 = UpSample(512, 256)
        self.up_convolution_3 = UpSample(256, 128)
        self.up_convolution_4 = UpSample(128, 64)

        self.out = nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1)

    def forward(self, x):
       down_1, p1 = self.down_convolution_1(x)
       down_2, p2 = self.down_convolution_2(p1)
       down_3, p3 = self.down_convolution_3(p2)
       down_4, p4 = self.down_convolution_4(p3)

       b = self.bottle_neck(p4)

       up_1 = self.up_convolution_1(b, down_4)
       up_2 = self.up_convolution_2(up_1, down_3)
       up_3 = self.up_convolution_3(up_2, down_2)
       up_4 = self.up_convolution_4(up_3, down_1)

       out = self.out(up_4)
       return out


In [ ]:
import os
from PIL import Image
from torch.utils.data.dataset import Dataset
from torchvision import transforms

class FloodSegmentationDataset(Dataset):
    def __init__(self, root_path, test=False):
        self.root_path = root_path
        if test:
              self.images = sorted([root_path+"/test/"+i for i in os.listdir(root_path+"/test/")])
              self.masks = sorted([root_path+"/test_masks/"+i for i in os.listdir(root_path+"/test_masks/")])
        else:
            self.images = sorted([root_path+"/test1/"+i for i in os.listdir(root_path+"/test1/")])
            self.masks = sorted([root_path+"/test1_masks/"+i for i in os.listdir(root_path+"/test1_masks/")])
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor()])

    def __getitem__(self, index):
        img = Image.open(self.images[index]).convert("RGB")
        mask = Image.open(self.masks[index]).convert("L")

        return self.transform(img), self.transform(mask)

    def __len__(self):
        return len(self.images)

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
import cv2

def single_image_inference(image_pth, model_pth, device):
    model = UNet(in_channels=3, num_classes=1).to(device)
    model.load_state_dict(torch.load(model_pth, map_location=torch.device(device)))

    # Open the image
    img_pil = Image.open(image_pth)

    # Convert RGBA to RGB if image has 4 channels
    if img_pil.mode == 'RGBA':
        img_pil = img_pil.convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor()])

    img = transform(img_pil).float().to(device)
    img = img.unsqueeze(0)

    pred_mask = model(img)

    img = img.squeeze(0).cpu().detach()
    img = img.permute(1, 2, 0)

    pred_mask = pred_mask.squeeze(0).cpu().detach()
    pred_mask = pred_mask.permute(1, 2, 0)
    pred_mask[pred_mask < 0] = 0
    pred_mask[pred_mask > 0] = 1

    if pred_mask.shape[-1] == 1:
        pred_mask = torch.cat((pred_mask, pred_mask, pred_mask), dim=-1)  # Convert to 3 channels if single channel

    pred_mask_np = pred_mask.numpy()
    resized_mask = cv2.resize(pred_mask_np, (640, 640), interpolation=cv2.INTER_NEAREST)

    plt.imsave("/content/sample_data/1.jpg", resized_mask, cmap="gray")

    fig = plt.figure()
    for i in range(1, 3):
        fig.add_subplot(1, 2, i)
        if i == 1:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(pred_mask, cmap="gray")

    #plt.savefig("/content/runs/detect/1.jpg")
    plt.show()


if __name__ == "__main__":
    SINGLE_IMG_PATH = "/content/sample_data/rhino1.jpg"
    DATA_PATH = "/content/drive/MyDrive/project/unet_seg/data"
    MODEL_PATH = "/content/drive/MyDrive/project/unet_seg/model/unet_test1_15.pth"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    #pred_show_image_grid(DATA_PATH, MODEL_PATH, device)
    single_image_inference(SINGLE_IMG_PATH, MODEL_PATH, device)


# YOLO

In [ ]:
  !pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/project/yolov8/best.pt') #replace path to best.pt

In [ ]:
# Pass an image path or a numpy array representing the image
results = model(SINGLE_IMG_PATH)  # Or results = model(image_array)
boxes = results[0].boxes

model.predict(SINGLE_IMG_PATH, save=True)

# **Combined Model**

In [ ]:
def flood_coverage_for_box(mask_image, bbox, threshold=200):
    coords = bbox[0]
    if hasattr(coords, "detach"):
        coords = coords.detach().cpu().tolist()
    x_center, y_center, width, height = coords

    x_min = int(max(0, x_center - width / 2))
    y_min = int(max(0, y_center - height / 2))
    x_max = int(min(mask_image.shape[1], x_center + width / 2))
    y_max = int(min(mask_image.shape[0], y_center + height / 2))

    if x_min >= x_max or y_min >= y_max:
        return 0.0, (x_min, y_min, x_max, y_max)

    box_mask = mask_image[y_min:y_max, x_min:x_max]
    if len(box_mask.shape) == 3:
        box_mask = cv2.cvtColor(box_mask, cv2.COLOR_BGR2GRAY)

    coverage = (box_mask > threshold).mean()
    return float(coverage), (x_min, y_min, x_max, y_max)


image1_path = SINGLE_IMG_PATH
image2_path = "/content/sample_data/1.jpg"
min_flood_coverage = 0.15

image = cv2.imread(image1_path)
flood_mask = cv2.imread(image2_path)

if image is None:
    raise FileNotFoundError(image1_path)
if flood_mask is None:
    raise FileNotFoundError(image2_path)

image_with_boxes = image.copy()
detections_in_flood = []

for box in boxes:
    coverage, (x_min, y_min, x_max, y_max) = flood_coverage_for_box(flood_mask, box.xywh)
    detections_in_flood.append(coverage)

    if coverage >= min_flood_coverage:
        cv2.rectangle(image_with_boxes, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(
            image_with_boxes,
            f"flood overlap: {coverage:.2f}",
            (x_min, max(0, y_min - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )

cv2.imwrite("/content/sample_data/high.jpg", image_with_boxes)
print(detections_in_flood)